# 02. SQL Order By, Sorting & Pagination: Beginner Guide

### 📝 SQL Execution Order:
```text
┌─ SQL Query Execution Order (Sequential Pipeline) ────────────────────────────┐
│ 1. FROM & JOIN (Load)    ➔ 2. WHERE (Filter)       ➔ 3. GROUP BY (Bucket)    │
│ ➔ 4. HAVING (Agg Filter) ➔ 5. SELECT (Pick Cols)   ➔ 6. DISTINCT (Dedup)     │
│ ➔ 7. ORDER BY (Sort)     ➔ 8. LIMIT / OFFSET (Page)                          │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **02. SQL Order By, Sorting & Pagination**. Relational database tables represent unordered mathematical relations. Unless an explicit `ORDER BY` clause is declared, the output sequence of returned rows is non-deterministic. This notebook covers sequential sorting (`ASC`/`DESC`), multi-column priority hierarchies, null boundary placement (`NULLS FIRST`/`NULLS LAST`), memory-safe result throttling (`LIMIT`/`TOP`), offset-based pagination, and modern Keyset (cursor-based) pagination.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Deterministic Sorting: `ORDER BY col ASC | DESC`
- [x] 🔹 Multi-Column Hierarchical Sorting: `ORDER BY col1 ASC, col2 DESC`
- [x] 🔹 Null Placement Control: `NULLS FIRST` vs `NULLS LAST`
- [x] 🔹 Result Throttling: `LIMIT N`
- [x] 🔹 Offset-Based Pagination: `LIMIT N OFFSET M`
- [x] 🔹 High-Performance Keyset (Cursor) Pagination: `WHERE id > last_seen_id ORDER BY id`
- [x] 🔍 Scenario: High-Throughput REST API Paginated Transaction Feed








In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Sequential Sorting: `ORDER BY ASC | DESC`
- **What it does:** Orders output tuples along one or more sort keys in ascending (`ASC`, default) or descending (`DESC`) order.
- **Syntax:** `SELECT * FROM table_name ORDER BY column_name [ASC | DESC]`
- **Dataset Application & Code Demonstration:** Sorts transactions from highest amount to lowest amount.


In [2]:
%%sql
SELECT 
    transaction_id, 
    customer_id, 
    transaction_amount, 
    transaction_date
FROM transactions
WHERE transaction_amount IS NOT NULL
ORDER BY transaction_amount DESC
LIMIT 5;


,transaction_id,customer_id,transaction_amount,transaction_date
0,TX113073,C22899,1999.98,2025-04-27
1,TX101707,C30635,1999.85,2025-02-17
2,TX108075,C22224,1999.74,05/07/2025
3,TX110393,C80575,1999.52,2025-11-06 10:08:13
4,TX113287,C64524,1999.41,09-25-2025


### 🔹 Multi-Column Hierarchical Sorting: `ORDER BY col1, col2`
- **What it does:** Applies secondary and tertiary sorting criteria to resolve tie-breakers across the primary sort key.
- **Syntax:** `SELECT * FROM table_name ORDER BY col1 ASC, col2 DESC`
- **Dataset Application & Code Demonstration:** Sorts transactions primarily by region ascending, then by transaction amount descending.


In [3]:
%%sql
SELECT 
    region, 
    transaction_id, 
    transaction_amount, 
    card_type
FROM transactions
WHERE transaction_amount IS NOT NULL
ORDER BY region ASC, transaction_amount DESC
LIMIT 8;


,region,transaction_id,transaction_amount,card_type
0,East,TX112849,1913.72,Amex
1,East,TX110923,1882.23,MasterCard
2,East,TX112042,1871.15,MasterCard
3,East,TX109920,1835.03,Visa
4,East,TX112710,1823.22,Discover
5,East,TX108316,1749.71,Visa
6,East,TX108255,1722.70,MasterCard
7,East,TX113853,1720.52,MasterCard


### 🔹 Null Placement Control: `NULLS FIRST` vs `NULLS LAST`
- **What it does:** Explicitly dictates whether `NULL` values sort at the beginning or end of the result stream regardless of sort direction.
- **Syntax:** `SELECT * FROM table_name ORDER BY column_name DESC NULLS LAST`
- **Dataset Application & Code Demonstration:** Sorts transaction amounts ascending while pushing missing values to the end.


In [4]:
%%sql
SELECT 
    transaction_id, 
    transaction_amount, 
    customer_id
FROM transactions
ORDER BY 
    CASE WHEN transaction_amount IS NULL THEN 1 ELSE 0 END, 
    transaction_amount ASC
LIMIT 6;


,transaction_id,transaction_amount,customer_id
0,TX113257,5.02,C26199
1,TX111606,5.82,C93202
2,TX113457,5.90,C41963
3,TX102109,6.12,C42493
4,TX102759,6.19,C73788
5,TX100235,6.44,C47606


### 🔹 Offset-Based Pagination: `LIMIT N OFFSET M`
- **What it does:** Limits the output stream to $N$ rows after skipping the first $M$ rows.
- **Syntax:** `SELECT * FROM table_name ORDER BY id LIMIT N OFFSET M`
- **Dataset Application & Code Demonstration:** Retrieves Page 3 of transaction records (Page Size = 5, Offset = 10).


In [5]:
%%sql
SELECT 
    transaction_id, 
    customer_id, 
    transaction_amount
FROM transactions
ORDER BY transaction_id ASC
LIMIT 5 OFFSET 10;


,transaction_id,customer_id,transaction_amount
0,TX100009,C63955,789.52
1,TX100010,C26505,21.31
2,TX100011,C74032,151.70
3,TX100012,C23356,1206.91
4,TX100013,C63269,640.47


### 🔹 High-Performance Keyset (Cursor) Pagination
- **What it does:** Replaces costly `OFFSET` skips with indexed filter predicates (`WHERE id > :last_seen_id`), achieving constant $O(1)$ page retrieval times.
- **Syntax:** `SELECT * FROM table_name WHERE id > last_seen_id ORDER BY id ASC LIMIT N`
- **Dataset Application & Code Demonstration:** Fetches the next page of records starting after a known transaction ID cursor.


In [6]:
%%sql
SELECT 
    transaction_id, 
    customer_id, 
    transaction_amount
FROM transactions
WHERE transaction_id > 'TX_00010'
ORDER BY transaction_id ASC
LIMIT 5;


,transaction_id,customer_id,transaction_amount


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Keyset Pagination vs OFFSET under High Write Load
- **Objective:** Demonstrate how cursor-based pagination guarantees deterministic result sets during active transaction ingestion.
- **Approach:** Query consecutive pages using indexed cursor comparisons.


In [7]:
%%sql
SELECT 
    transaction_id,
    customer_id,
    transaction_amount,
    account_age_months
FROM transactions
WHERE transaction_amount > 1000.00
  AND transaction_id > 'TX_00050'
ORDER BY transaction_id ASC
LIMIT 5;


,transaction_id,customer_id,transaction_amount,account_age_months
